# Stream Lifecycle and Result Lists

CSC-239 · Module 10 · Lesson 3 of 4

You can build an ordered map/filter/toList pipeline. Now you will distinguish the pipeline description from its execution and from the list produced by that execution.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Distinguish a stream description, its terminal execution, and a reusable result list.
- Create fresh streams and a separate modifiable list without changing the original result.


## Why This Matters

A report may be prepared before all its source entries are present. After generating it, you may need a second computation or an editable working list. Each action has a different rule.


## Check Your Starting Point

Identify the source, intermediate operation, and terminal operation in a short pipeline. Recall the difference between copying a reference and creating a separate collection. Explain why changing a list and changing one object inside a list are different actions.

**My explanation:**


## Concept

### A pipeline describes work before it starts

**Lazy evaluation** means that the pipeline describes work before a terminal operation starts processing elements. A variable of type `Stream<String>` can hold that pending computation. Import Stream from java.util.stream when you declare such a variable.

For the ArrayList source used here, the stream is **late-bound**: it observes the source contents when terminal processing begins. Creating the stream does not take a frozen snapshot of the original list's elements.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("kit");
List<String> report = pending.toList();
for (String item : report) {
    System.out.println(item);
}
```

This prints Item: map and Item: kit. The source changes before the terminal operation starts. Both entries are present when toList begins processing.

This example does not change the source while processing it. Those are different situations. Do not add or remove source entries from a lambda running inside the pipeline.

The late-binding description here applies to the taught ArrayList source. Do not assume every way of creating a stream has the same source behavior.

### Use a stream for one computation

A **single-use stream** is intended for one pipeline computation. After a terminal operation, create a new stream if you need another computation. Saving the old reference does not reset its state.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
Stream<String> pending = source.stream();
List<String> report = pending.toList();
try {
    pending.count();
} catch (IllegalStateException exception) {
    System.out.println("Create a new stream.");
}
System.out.println("Fresh count: " + source.stream().count());
```

This prints Create a new stream. and Fresh count: 1. The caught IllegalStateException demonstrates the prohibited reuse in this example. It is evidence of an invalid operation, not a reason to use exception handling as the normal way to request a second computation.

The count terminal operation returns a long, Java's integer type with a wider range than int. Here println displays that numeric result directly. You do not need to convert it to int to print it.

The source collection remains available. So does the completed report list. Iterate over a result list again when you want to read the same report; create a new stream when you want a new computation over a source.

### Read an unmodifiable result

The previous lesson stated that toList returns an **unmodifiable result list**. Its elements can be read, but attempts to add, remove, or replace list entries are rejected.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
List<String> report = source.stream().toList();
try {
    report.add("kit");
} catch (UnsupportedOperationException exception) {
    System.out.println("Result cannot be edited.");
}
System.out.println("Result size: " + report.size());
```

This prints Result cannot be edited. and Result size: 1. UnsupportedOperationException reports that this list does not support the requested change. The result was not partly extended.

The List interface type does not promise that every implementation accepts all changes. Its operation contracts can describe unsupported operations. Select a collection implementation according to the operations the task needs.

### Create a separate collection for edits

A **collection copy** creates a new collection containing the current elements of another collection. The ArrayList constructor can accept an existing collection. This creates a modifiable list with a separate list structure.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
List<String> report = source.stream().toList();
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("kit");
System.out.println("Report: " + report.size());
System.out.println("Editable: " + editable.size());
System.out.println("Source: " + source.size());
```

This prints Report: 1, Editable: 2, and Source: 1. Adding an entry to editable changes that list's structure. It does not add an entry to the report or the source.

The copy is **shallow**: its entries initially contain the same element references. The constructor does not make new copies of the element objects. That follows the reference-copying rules from Module 5. These examples use String elements, so the task changes list entries without changing a mutable object shared by two lists.

Compare this constructor call with assigning editable = anotherList. Assignment alone copies a reference and can create an alias to the same list. The new ArrayList expression creates a separate list object.

### Keep the three stages distinct

| Thing | What it represents | Appropriate next action |
| --- | --- | --- |
| Source ArrayList | Stored input elements | Create a fresh stream for another computation |
| Pending Stream | One computation not yet completed | Invoke one terminal operation |
| Result List | Elements produced by a completed computation | Read again, or copy into a modifiable collection |

An empty source is still a valid test. It produces an empty result and an empty editable copy. Adding a new entry to that copy should leave the other two sizes at zero.

Use returned values and completed results as evidence. Do not rely on printing inside an intermediate lambda to prove an exact execution schedule; stream implementations may avoid work that cannot affect the terminal result.


## Video Demonstration

Watch when the pending pipeline observes its ArrayList source. Then compare the sizes of the report, its editable copy, and the source.

<video controls preload="metadata" width="960">
  <source src="media/03_stream_lifecycle_and_result_lists/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_stream_lifecycle_and_result_lists/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the stream lifecycle and result lists demonstration transcript](media/03_stream_lifecycle_and_result_lists/transcript.md).


## Worked Example

**Subgoal 1: prepare a pending computation.** Map the source values without yet invoking a terminal operation.

**Subgoal 2: execute once.** Add the second source entry before toList starts, then collect the report.

**Subgoal 3: choose the next operation correctly.** Copy the report for edits and create a fresh stream for a new count.


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("kit");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());


Expected output:

```text
Reported: 2
Original result: 2
Editable copy: 3
Fresh count: 2
```

The ArrayList contains map and kit when toList begins. The report therefore contains two entries. Adding a pen entry to a new ArrayList changes only that copy, so the report stays at two and the copy grows to three. A fresh stream counts the two source entries.


## Predict, Run, Trace, and Explain

### Predict the report, copy and fresh count

Before running, predict all four printed lines. Identify which source entries exist when toList starts and which list receives the pen entry. Explain whether creating pending immediately freezes the source entries and whether the final count uses pending again.

My four predicted lines:

Source entries present when terminal processing starts:

Which list receives the additional entry:

Whether the final count reuses pending:


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());


Run the complete prediction program in the Workspace. Retain your original prediction and write a post-run explanation of each confirmed or corrected line. Identify the pending stream, the completed report and the editable copy. Explain why the source count can differ from the editable copy’s size.

My original prediction:

Actual four lines:

My explanation of each result:

Why the source and copy are counted separately:

### Trace creation, execution and reuse

Trace the program at these points: after creating pending, just before toList, just after toList, after creating editable, and after adding the pen entry. Record the source contents and, where they exist, the report and copy contents. Mark the terminal operation that consumes pending. State what can be read again afterward, what needs a fresh stream, and what the collection copy does and does not duplicate.

| Point | Source contents | Report if created | Copy if created |
| --- | --- | --- | --- |
| Pending created | | | |
| Before toList | | | |
| After toList | | | |
| Copy created | | | |
| Copy edited | | | |
Terminal operation using pending:

Reusable result versus fresh computation:

What the collection constructor copies:

<details>
<summary>Show answer</summary>

Creating pending describes the mapping without collecting a report. The ArrayList holds badge, card and key when toList begins, so the report contains Item: badge, Item: card and Item: key. Reported is three. The ArrayList constructor copies those entries into a separate editable list. Adding Item: pen changes only that copy, leaving the report size three and making the copy size four. The final count uses a fresh stream over the original three source entries and reports three. count returns long; printing it does not require changing it to int. Before toList, pending refers to the uncompleted computation. After toList, that stream has been used, while report holds a list that can be read again. The collection constructor creates a separate list structure but initially copies the same String references; it does not create new String objects. This is the shallow copy described in the reading. The demonstrated source timing belongs to this ArrayList source, with changes before or after terminal processing rather than inside it.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
```

Common error: Counting only the entry present when pending was created. Adding the copy’s new entry to the report or source count. Treating a fresh stream as another use of the old stream.

</details>


### Read a report again and request a fresh computation

This complete program catches an intentionally attempted reuse of a consumed stream. Predict all printed lines before running. Explain the different results of pending.count(), reading report in the loop and with get, and source.stream().count(). After running, retain your prediction and explain why reading the stored result again is allowed even though the old stream cannot supply another computation.

My predicted lines:

Actual output and my post-run explanation:

Why the old stream call is rejected:

Why the two report reads and fresh count are allowed:


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
Stream<String> pending = source.stream().map(item -> "Tree: " + item);
source.add("elm");
List<String> report = pending.toList();
System.out.println("First report: " + report.size());
try {
    pending.count();
} catch (IllegalStateException problem) {
    System.out.println("Fresh stream needed.");
}
for (String item : report) {
    System.out.println(item);
}
System.out.println("Read again: " + report.get(0));
System.out.println("Fresh count: " + source.stream().count());


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The pending stream is collected once, producing the two Tree entries. Calling count on that same stream is rejected with IllegalStateException and the catch block prints Fresh stream needed. The completed report remains usable: the loop reads both entries and get reads its first entry again. The final count succeeds because source.stream() creates a new stream. The caught error demonstrates the invalid reuse; requesting a fresh stream directly is the normal way to perform another computation.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
Stream<String> pending = source.stream().map(item -> "Tree: " + item);
source.add("elm");
List<String> report = pending.toList();
System.out.println("First report: " + report.size());
try {
    pending.count();
} catch (IllegalStateException problem) {
    System.out.println("Fresh stream needed.");
}
for (String item : report) {
    System.out.println(item);
}
System.out.println("Read again: " + report.get(0));
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
First report: 2
Fresh stream needed.
Tree: oak
Tree: elm
Read again: Tree: oak
Fresh count: 2
```

Common error: Expecting the consumed stream to restart automatically. Treating a list read as another terminal operation on pending. Using the catch block as the normal method for obtaining another result.

</details>


### Compare rejected result edits with an editable copy

Predict all printed lines before running this complete caught-error example. Explain why removing the first report entry is rejected but removing the first copy entry succeeds. Then run two separate complete comparisons: replace only report.remove(0) with `report.add("Tree: ash")`, and then with `report.set(0, "Tree: ash")`. Keep editable.remove(0) unchanged in both. Predict before each run, record actual output and explain whether the report was partly changed.

My remove-case prediction:

Actual output and explanation:

My add-case prediction and actual output:

My replacement-case prediction and actual output:

Why report and editable have different editing behavior:


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.remove(0);
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The remove call on report is rejected with UnsupportedOperationException, so the catch block prints Copy before editing. The report still begins with Tree: oak. A new ArrayList holds a separate copy of the report’s entries. Removing its first entry leaves Tree: elm in the copy while the original report stays at size two. The copy has size one. The add and set comparison programs are also rejected when they target report; each then performs the same successful removal on a newly created editable copy. Changing the declared reference type alone would not make the original list support edits.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.remove(0);
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());
```

Expected output:

```text
Copy before editing.
Report first: Tree: oak
Copy first: Tree: elm
Report size: 2
Copy size: 1
```

Common error: Assuming List guarantees that every implementation supports edits. Assuming a caught rejected edit partially changed the result. Assigning another reference to report instead of constructing a new list.

**Check case 2.** The attempted addition is rejected before an entry is added to report. The later ArrayList copy is still made from the two original entries, and its removal leaves Tree: elm.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.add("Tree: ash");
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());
```

Expected output:

```text
Copy before editing.
Report first: Tree: oak
Copy first: Tree: elm
Report size: 2
Copy size: 1
```

**Check case 3.** The attempted replacement is rejected, so report still starts with Tree: oak. The independent editable copy accepts removal and retains only Tree: elm.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.set(0, "Tree: ash");
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());
```

Expected output:

```text
Copy before editing.
Report first: Tree: oak
Copy first: Tree: elm
Report size: 2
Copy size: 1
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the stream and copy choices

Replace PENDING_TYPE, FINISH, COPY_FROM and NEW_STREAM in the displayed program. Choose the pending stream type, the operation that produces the report, the collection passed to the copy constructor, and the method that creates a fresh stream for counting. Use `Stream<String>`, toList, report and stream in the matching places. Put the complete finished program in the work cell and run it. Check the four prediction results again and explain why each choice fits its object’s role.

This sample is for repair:

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
PENDING_TYPE pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.FINISH();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(COPY_FROM);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.NEW_STREAM().count());
```


My four replacements and reasons:

Actual output:

Why the copy argument and fresh-stream call refer to different collections:

<details>
<summary>Show answer</summary>

Use `Stream<String>` for PENDING_TYPE, toList for FINISH, report for COPY_FROM and stream for NEW_STREAM. These choices preserve the difference between the pending computation, its completed list and a new list structure for edits. The last expression asks the source for a new stream rather than invoking another terminal operation on pending. Creating pending describes the mapping without collecting a report. The ArrayList holds badge, card and key when toList begins, so the report contains Item: badge, Item: card and Item: key. Reported is three. The ArrayList constructor copies those entries into a separate editable list. Adding Item: pen changes only that copy, leaving the report size three and making the copy size four. The final count uses a fresh stream over the original three source entries and reports three. count returns long; printing it does not require changing it to int.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
```

Common error: Using a list type for the pending computation. Copying source when the task needs mapped report entries. Trying to use pending again for the final count.

</details>


### Move source changes after collection

Move only `source.add("key")` so it runs immediately after report is collected and before the Reported print. Predict the four lines, then run the complete program. Explain which collections can now include key. Next move the card addition after collection too, placing it before the moved key addition. Predict and run again. Keep the mapping, copy edit and fresh count unchanged. Explain why a stored report does not acquire later source additions.


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());


My prediction after moving key:

Actual output and timing explanation:

My prediction after moving both card and key:

Actual output and why the report stays unchanged:

<details>
<summary>Show answer</summary>

With key added after toList, terminal processing observes badge and card only. The report therefore has two entries; the copy grows from two to three when Item: pen is added. The source later contains all three entries, so the fresh count is three. Moving card after collection as well leaves only badge in the generated report. That report stays at one, its copy grows to two, and the source still reaches three. These changes occur after processing, not during it.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
List<String> report = pending.toList();
source.add("key");
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 2
Original result: 2
Editable copy: 3
Fresh count: 3
```

Common error: Assuming a previously produced report automatically receives later source entries. Moving the source edits into the mapping lambda. Changing the fresh count to reuse the consumed stream.

**Additional test: `Both card and key are added after collection`.** Only badge is present when toList starts. The report contains one mapped entry; adding Item: pen grows the copy to two. Both later source additions are included in the fresh count of three.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
List<String> report = pending.toList();
source.add("card");
source.add("key");
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 1
Original result: 1
Editable copy: 2
Fresh count: 3
```

</details>


### Repair an alias that cannot accept edits

The displayed program intends editable to be an independent list that accepts changes. Find the statement that creates another reference to report instead of a new collection. Predict how far the program can run and which operation is rejected. Replace that declaration with a suitable ArrayList constructor while keeping the source, mapping and print statements unchanged. Put only the complete repaired program in the work cell and run it. Then add `editable.set(0, "Item: changed")` immediately after the copy’s pen addition, and print report.get(0) and editable.get(0) after the count. Predict both first entries and explain the actual result. Use the labels `"Original first: "` and `"Copy first: "`.

This sample is for repair:

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
List<String> editable = report;
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```


The faulty reference assignment and rejected operation:

My prediction of output before the failure:

My repaired declaration and actual output:

My prediction and actual first-entry checks:

Why a separate list structure matters:

<details>
<summary>Show answer</summary>

The faulty assignment makes editable refer to the same unmodifiable list as report. The program prints `Reported: 3`, then `editable.add("Item: pen")` throws `UnsupportedOperationException`. The three later count lines are not reached. A new variable name does not create a new list or change its editing rules. The repair uses `new ArrayList<String>(report)` to create a separate list structure. Adding an entry to that copy succeeds and leaves the report unchanged. Replacing index zero in the repaired copy also changes only that copy’s entry: report still begins with Item: badge while the copy begins with Item: changed. The constructor initially copies element references, but set replaces one list entry rather than changing the shared String object.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
```

Common error: Changing the reference type without creating a new collection. Editing report when only the working copy should change. Claiming the constructor makes new copies of all element objects.

**Additional test: `Replace the first entry only in the repaired copy`.** The original report retains Item: badge at index zero while the copy now holds Item: changed there. The earlier added pen still makes the copy size four. Replacing a copy entry does not change source or report entries.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
editable.set(0, "Item: changed");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
System.out.println("Original first: " + report.get(0));
System.out.println("Copy first: " + editable.get(0));
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
Original first: Item: badge
Copy first: Item: changed
```

</details>


## Independent Practice

### Build an editable room report

Create an ArrayList named source containing lab. Build a pending stream that maps each room to `"Room: "` followed by its name, then add hall and desk to source before terminal execution. Collect report with toList. Copy report into an ArrayList named editable, remove index zero from the copy, and print its remaining entries in order. Print `"Original result: "` with report size, `"Editable copy: "` with copy size and `"Fresh count: "` using a new stream over source. The baseline prints Room: hall, Room: desk, Original result: 3, Editable copy: 2 and Fresh count: 3 on separate lines. Include all three imports and complete setup. Explain why the removal belongs on the copy.

My complete program:

Actual baseline output:

Source entries observed when toList starts:

Why the copy can change while report remains readable and unchanged:

Why counting requests a fresh stream:


### Check empty and one-entry sources

For boundary tests, replace the copy’s removal with an if statement that calls remove(0) only when editable.size() is greater than zero. First test a source with no room additions. Next test a source containing only lab. Finally return to the empty source and add `"Room: studio"` to editable after the guarded removal but before printing. Keep the pipeline and result-reading statements in each complete program. Predict every output line before each run and explain the actual results afterward. Compare the source, report and copy sizes, then restore and rerun the original three-room baseline. Explain why an empty copy cannot remove index zero and why adding to it should leave the other two collections empty.

For each case, my predicted lines:

Actual empty-source output and explanation:

Actual one-room output and explanation:

Actual empty-copy addition output and explanation:

Restored baseline output:


<details>
<summary>Show answer</summary>

The ArrayList contains lab, hall and desk before toList starts. Mapping produces Room: lab, Room: hall and Room: desk in that order. The new ArrayList copies the three result entries into a separate editable list. Removing index zero from the copy leaves Room: hall and Room: desk to print. The report still has three entries, the copy has two, and a fresh stream counts three source entries. The original report is unmodifiable, so the removal belongs on the copy. Copying the list structure initially copies element references; these String values are not changed by the removal. For an empty source, toList and the copy constructor produce empty lists. The guard skips removal because index zero is absent; all three sizes are zero. With only lab, the report has one entry and the copy’s allowed removal leaves it empty. Adding Room: studio to a copy made from an empty report creates one copy entry while source and report remain empty. These tests change the supplied input or copy while retaining a fresh stream for each new count.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("lab");
Stream<String> pending = source.stream().map(room -> "Room: " + room);
source.add("hall");
source.add("desk");
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Room: hall
Room: desk
Original result: 3
Editable copy: 2
Fresh count: 3
```

Common error: Collecting before adding the required hall and desk entries. Removing from the unmodifiable report. Reusing the consumed pending stream for the count.

**Additional test: Empty source with guarded removal.** There are no source additions. Both produced lists are empty. The guard skips removal and the printing loop has no entries, but all three zero-size lines still run.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
Stream<String> pending = source.stream().map(room -> "Room: " + room);
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
if (editable.size() > 0) {
    editable.remove(0);
}
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Original result: 0
Editable copy: 0
Fresh count: 0
```

**Additional test: Only lab with guarded removal.** The source and report contain one room. Removing the only copied entry is valid and leaves no copy entry to print. The fresh source count remains one.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("lab");
Stream<String> pending = source.stream().map(room -> "Room: " + room);
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
if (editable.size() > 0) {
    editable.remove(0);
}
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Original result: 1
Editable copy: 0
Fresh count: 1
```

**Additional test: Add a room only to the initially empty copy.** The guarded empty removal does nothing. Adding Room: studio afterward grows only the independent copy, so its one room is printed while source and original result stay at size zero.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
Stream<String> pending = source.stream().map(room -> "Room: " + room);
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
if (editable.size() > 0) {
    editable.remove(0);
}
editable.add("Room: studio");
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Room: studio
Original result: 0
Editable copy: 1
Fresh count: 0
```

</details>


## Summary

A pending pipeline describes work. A terminal operation starts processing and consumes that stream. An ArrayList stream observes source entries when terminal processing begins, so changing the source before execution differs from changing it during execution. A toList result can be read repeatedly but rejects entry changes. A new ArrayList copies its entries into a separate modifiable list structure.

Close the answers and explain why the source, stream, and result have different next actions.


## Reflection

Describe a report that needs a stable generated result and a separate editable draft. Explain when you would generate a new report and when you would copy an existing report. Identify one mistake caused by confusing a pending computation with a stored result.

**My design and explanation:**

Next, you will combine elements into one result and compare safe sequential and parallel computations.


## Supplemental Reading

- [Stream lifecycle and toList](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/Stream.html) describes single-use streams, terminal operations, and result-list restrictions.
- [Collection stream sources](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/Collection.html#stream()) explains source binding and interference expectations.
- [ArrayList constructors](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/ArrayList.html) documents copying collection entries into a new list.
